# JEPX Pipeline Validation Notebook

This notebook validates end-to-end outputs by reading:
- RustFS raw object (`raw/jepx/spot_summary/...`)
- Bronze table (`s3://<bucket>/bronze/jepx_spot_price`)
- Silver table (`s3://<bucket>/silver/silver_jepx_spot_price_base`)

Each section displays the first 5 rows.

## 1. Imports

In [ ]:
from __future__ import annotations

import io
import sys
from pathlib import Path

import duckdb
import polars as pl

WORKSPACE_ROOT = (
    Path.cwd().resolve().parents[1] if Path.cwd().name == "Jupyter" else Path("/workspace")
)
SRC_PATH = WORKSPACE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from core.storage_client import RustFSClient, RustFSConfig

## 2. Set configs

In [ ]:
BUCKET_NAME = "jp-power-grid-dev"
RAW_PREFIX = "raw/jepx/spot_summary/"
RAW_ENCODING = "cp932"

BRONZE_ICEBERG_PATH = f"s3://{BUCKET_NAME}/bronze/jepx_spot_price"
SILVER_TABLE_NAME = "main_silver.silver_jepx_spot_price_base"
DBT_DUCKDB_PATH = "/workspace/src/dbt/jepx_power/jepx_power.duckdb"

## 3. Define functions

In [ ]:
rustfs = RustFSClient()
storage_options = rustfs.get_storage_options()

In [ ]:
def create_s3_duckdb_connection(config: RustFSConfig) -> duckdb.DuckDBPyConnection:
    conn = duckdb.connect(database=":memory:")
    conn.execute("INSTALL httpfs;")
    conn.execute("LOAD httpfs;")
    conn.execute("INSTALL iceberg;")
    conn.execute("LOAD iceberg;")

    conn.execute(
        f"SET s3_endpoint='{config.endpoint_url.replace('http://', '').replace('https://', '')}';"
    )
    conn.execute("SET s3_url_style='path';")
    conn.execute("SET s3_use_ssl=false;")
    conn.execute(f"SET s3_access_key_id='{config.access_key}';")
    conn.execute(f"SET s3_secret_access_key='{config.secret_key}';")
    conn.execute(f"SET s3_region='{config.region}';")
    conn.execute("SET unsafe_enable_version_guessing=true;")
    return conn


def read_latest_raw_head(bucket_name: str, prefix: str, rows: int = 5) -> pl.DataFrame:
    raw_objects = sorted(rustfs.list_files(bucket_name=bucket_name, prefix=prefix))
    if not raw_objects:
        raise RuntimeError(f"No raw objects found under s3://{bucket_name}/{prefix}")

    latest_key = raw_objects[-1]
    binary = rustfs.get_object(bucket_name=bucket_name, object_name=latest_key)
    decoded = binary.decode(RAW_ENCODING)
    df = pl.read_csv(io.StringIO(decoded), infer_schema_length=0)
    print(f"Using raw object: s3://{bucket_name}/{latest_key}")
    return df.head(rows)


def read_iceberg_head(
    conn: duckdb.DuckDBPyConnection, table_path: str, rows: int = 5
) -> pl.DataFrame:
    query = f"SELECT * FROM iceberg_scan('{table_path}') LIMIT {rows};"
    return conn.execute(query).pl()


def read_duckdb_table_head(db_path: str, table_name: str, rows: int = 5) -> pl.DataFrame:
    conn = duckdb.connect(database=db_path, read_only=True)
    try:
        return conn.execute(f"SELECT * FROM {table_name} LIMIT {rows};").pl()
    finally:
        conn.close()

In [ ]:
s3_conn = create_s3_duckdb_connection(RustFSConfig.from_env())

In [9]:
raw_head = read_latest_raw_head(BUCKET_NAME, RAW_PREFIX, rows=5)
raw_head

2026-05-11 14:57:03,208 | INFO | Files in jp-power-grid-dev/raw/jepx/spot_summary/: ['raw/jepx/spot_summary/spot_summary_2005.csv', 'raw/jepx/spot_summary/spot_summary_2006.csv', 'raw/jepx/spot_summary/spot_summary_2007.csv', 'raw/jepx/spot_summary/spot_summary_2008.csv', 'raw/jepx/spot_summary/spot_summary_2009.csv', 'raw/jepx/spot_summary/spot_summary_2010.csv', 'raw/jepx/spot_summary/spot_summary_2011.csv', 'raw/jepx/spot_summary/spot_summary_2012.csv', 'raw/jepx/spot_summary/spot_summary_2013.csv', 'raw/jepx/spot_summary/spot_summary_2014.csv', 'raw/jepx/spot_summary/spot_summary_2015.csv', 'raw/jepx/spot_summary/spot_summary_2016.csv', 'raw/jepx/spot_summary/spot_summary_2017.csv', 'raw/jepx/spot_summary/spot_summary_2018.csv', 'raw/jepx/spot_summary/spot_summary_2019.csv', 'raw/jepx/spot_summary/spot_summary_2020.csv', 'raw/jepx/spot_summary/spot_summary_2021.csv', 'raw/jepx/spot_summary/spot_summary_2022.csv', 'raw/jepx/spot_summary/spot_summary_2023.csv', 'raw/jepx/spot_summary

2026-05-11 14:57:03,222 | INFO | Object raw/jepx/spot_summary/spot_summary_2026.csv retrieved from jp-power-grid-dev


Using raw object: s3://jp-power-grid-dev/raw/jepx/spot_summary/spot_summary_2026.csv


受渡日,時刻コード,売り入札量(kWh),買い入札量(kWh),約定総量(kWh),システムプライス(円/kWh),エリアプライス北海道(円/kWh),エリアプライス東北(円/kWh),エリアプライス東京(円/kWh),エリアプライス中部(円/kWh),エリアプライス北陸(円/kWh),エリアプライス関西(円/kWh),エリアプライス中国(円/kWh),エリアプライス四国(円/kWh),エリアプライス九州(円/kWh),売りブロック入札総量(kWh),売りブロック約定総量(kWh),買いブロック入札総量(kWh),買いブロック約定総量(kWh)
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2026/04/01""","""1""","""27474550""","""22189150""","""17741050""","""17.75""","""17.75""","""20.15""","""20.15""","""20.61""","""9.47""","""9.47""","""9.47""","""9.47""","""8.78""","""8938850""","""3146000""","""1198550""","""823850"""
"""2026/04/01""","""2""","""28671000""","""21485150""","""18046400""","""14.83""","""17.45""","""19.44""","""19.44""","""20.57""","""8.58""","""8.58""","""8.58""","""8.58""","""8.58""","""11120400""","""4151950""","""1179050""","""821800"""
"""2026/04/01""","""3""","""29196500""","""21271000""","""18374350""","""12.70""","""16.54""","""14.83""","""14.83""","""14.83""","""14.83""","""14.83""","""14.83""","""14.83""","""8.78""","""12131200""","""4710500""","""1239600""","""840700"""
"""2026/04/01""","""4""","""29509550""","""21469950""","""18377150""","""13.21""","""15.45""","""15.45""","""15.45""","""15.45""","""15.45""","""15.45""","""15.45""","""15.45""","""8.78""","""12709150""","""4865400""","""1258200""","""840400"""
"""2026/04/01""","""5""","""29512150""","""21821200""","""18440350""","""14.83""","""16.07""","""16.07""","""16.07""","""16.07""","""16.07""","""16.07""","""16.07""","""16.07""","""9.03""","""12751250""","""4859650""","""1257850""","""841250"""


In [10]:
bronze_head = read_iceberg_head(s3_conn, BRONZE_ICEBERG_PATH, rows=5)
silver_head = read_duckdb_table_head(DBT_DUCKDB_PATH, SILVER_TABLE_NAME, rows=5)

print("Bronze table head")
bronze_head

print("Silver table head")
silver_head

Bronze table head
Silver table head


delivery_datetime,selling_bid_volume,purchase_bid_volume,contracted_volume,system_price
"datetime[μs, Etc/UTC]",i64,i64,i64,i64
2007-02-06 12:30:00 UTC,929000,261500,50000,10
2007-02-07 03:30:00 UTC,709000,202500,83500,6
2007-02-07 06:00:00 UTC,706500,215500,79500,6
2007-02-07 09:00:00 UTC,946500,273500,87000,9
2007-02-07 22:30:00 UTC,754000,258500,70000,7
